In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Font Configuration for Colab
plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

# [Important] Specify the Pdataset folder path
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/subway_project/'
FILE_PATH = os.path.join(BASE_PATH, 'Seoul_subway_data_2023_2026.csv')

print("데이터 경로 지정 완료:", FILE_PATH)

데이터 경로 지정 완료: /content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/subway_project/Seoul_subway_data_2023_2026.csv


In [18]:
import pandas as pd

# Please update the path to match your Google Drive path.
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/subway_project/'

# Load data split by tab (\t) and UTF-8 encoding (no header)
subway_df = pd.read_csv(BASE_PATH + 'Seoul_subway_data_2023_2026.csv', sep='\t', encoding='cp949', header=None)

print("데이터 총 행/열 크기:", subway_df.shape)
print("\n--- 상위 3개 행 데이터 모양 데이터 확인 ---")
subway_df.head(3)

데이터 총 행/열 크기: (25393, 51)

--- 상위 3개 행 데이터 모양 데이터 확인 ---


,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
0,202301,1호선,동대문,717,11,9399,1791,7814,5063,11183,...,4311,7667,529,2104,5,254,0,0,0,0
1,202301,1호선,동묘앞,176,5,2429,929,3063,4244,5788,...,1264,3400,80,1320,3,330,0,0,0,0
2,202301,1호선,서울역,555,27,6560,7430,11059,41800,36434,...,21933,12316,3388,2777,69,235,0,5,0,0


In [13]:
# 기본 컬럼명 설정 (0: 사용월, 1: 호선, 2: 역명)
col_names = ['사용월', '호선', '역명']

# 3번 컬럼부터는 04시승차, 04시하차, 05시승차, 05시하차... 형태로 자동 생성
for hour in range(4, 28):
    h = hour if hour < 24  else hour - 24 # 24시 이후는 00시, 01시로 표현
    col_names.extend([f'{h:02d}시승차', f'{h:02d}시하차'])

# 실제 데이터 열 개수에 맞게 슬라이싱하여 컬럼명 적용
subway_df.columns = col_names[:len(subway_df.columns)]

print("컬럼명 변경 완료! 변경된 컬럼 예시:")
print(subway_df.columns[:10].tolist())

컬럼명 변경 완료! 변경된 컬럼 예시:
['사용월', '호선', '역명', '04시승차', '04시하차', '05시승차', '05시하차', '06시승차', '06시하차', '07시승차']


In [14]:
# Please update the path to match your Google Drive path.
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/Pdataset/'

# 1. Load subway data
# Since the first line is not a header, assign arbitrary names or handle it later
subway_raw = pd.read_csv(BASE_PATH + 'Seoul_subway_data_2023_2026.csv', sep='\t', encoding='cp949', header=None)

# 2. 나머지 데이터 로드 예시 (실제 다운로드한 파일명으로 매칭하세요)
# subway_geo = pd.read_csv(BASE_PATH + '지하철역_좌표데이터.csv', encoding='cp949')
# bus_raw = pd.read_csv(BASE_PATH + '버스정류소_승하차데이터.csv', encoding='cp949')
# bus_geo = pd.read_csv(BASE_PATH + '버스정류소_위치정보.csv', encoding='cp949')
# shop_raw = pd.read_csv(BASE_PATH + '소상공인_상가정보_서울.csv', encoding='cp949')

print("데이터 로드 준비 완료! 지하철 데이터 크기:", subway_raw.shape)

데이터 로드 준비 완료! 지하철 데이터 크기: (25393, 51)


In [15]:
# [Step 3 수정본] 일평균 총 하차 인원 기반 전처리
import pandas as pd
import re

# 1. 데이터 로드
# 파일 경로를 BASE_PATH 변수를 활용하도록 수정합니다.
subway_raw = pd.read_csv(BASE_PATH + 'Seoul_subway_data_2023_2026.csv', sep='\t', encoding='cp949', header=None)

# 2. 짝수 인덱스 컬럼(하차 컬럼)만 골라내서 전체 하루 총 하차량 구하기
# 데이터 구조가 [사용월, 호선, 역명, 04승, 04하, 05승, 05하...] 이므로
# 4번 열부터 시작해서 2칸씩 건너뛰는 열들이 전부 '하차' 데이터입니다.
desc_cols = subway_raw.columns[4::2]

# Sum all time-slot alighting population to create 'total_daily_alighting'
subway_raw['하루총하차'] = subway_raw[desc_cols].sum(axis=1)

# 3. Clean station names
def clean_station(name):
    name = re.sub(r'\([^)]*\)', '', str(name))
    if name.endswith('역') and name != '서울역':
        name = name[:-1]
    return name.strip()

subway_raw['정제역명'] = subway_raw[2].apply(clean_station) # 2번 열이 역명

# 4. Filter for latest month (e.g. May 2026) or calculate average by station
subway_summary = subway_raw.groupby(['정제역명'])['하루총하차'].mean().reset_index()

print("전처리 완료! 이제 역별 '일평균 총 하차인원'으로 단순화되었습니다.")
subway_summary.head()

전처리 완료! 이제 역별 '일평균 총 하차인원'으로 단순화되었습니다.


,정제역명,하루총하차
0,4.19민주묘지,101026.268293
1,가능,184250.682927
2,가락시장,259859.817073
3,가산디지털단지,845839.829268
4,가양,625226.414634


In [16]:
from math import radians, cos, sin, asin, sqrt

# 두 좌표 사이의 거리를 계산하는 하버사인(Haversine) 함수
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371 # 지구 반지름(km)
    return c * r * 1000 # 미터 단위 반환

print("공간 결합 함수 정의 완료! 반경 500m 이내의 버스/상가를 이 함수로 필터링합니다.")

공간 결합 함수 정의 완료! 반경 500m 이내의 버스/상가를 이 함수로 필터링합니다.


In [ ]:
import folium

# Create mock final merged DataFrame (actual data should match this structure)
demo_data = {
    '역명': ['망원', '홍대입구', '뚝섬', '성수', '강남', '상수'],
    '위도': [37.5560, 37.5575, 37.5472, 37.5446, 37.4980, 37.5477],
    '경도': [126.9015, 126.9245, 127.0430, 127.0559, 127.0276, 126.9229],
    '총하차인원': [12000, 85000, 15000, 68000, 95000, 9800],     # 지하철 + 버스 밤 하차 합계
    '맛집수': [180, 520, 140, 480, 610, 110]                     # 반경 500m 주점/음식점 수
}
final_df = pd.DataFrame(demo_data)

# 💡 Feature Engineering: Calculate Congestion Cost-Benefit (C/P) Index
final_df['혼잡가성비'] = (final_df['맛집수'] / final_df['총하차인원']) * 10000

# 지도 시각화 시작
seoul_map = folium.Map(location=[37.5502, 126.982], zoom_start=12, tiles='CartoDB positron')

for idx, row in final_df.iterrows():
    # Decide color based on C/P ratio (thresholds can be adjusted)
    if row['혼잡가성비'] > 70:
        color = 'green'  # 🟢 가성비 최고 (숨은 꿀역)
    elif row['총하차인원'] > 60000:
        color = 'red'    # 🔴 교통 헬게이트 및 대형 핫플
    else:
        color = 'blue'   # 🔵 무난한 일반 상권

    # Create formatted HTML popup info
    popup_text = f"""
    <div style='width:200px;'>
        <h4><b>{row['역명']}역</b></h4>
        <hr style='margin:5px 0;'>
        <b>주변 맛집 수:</b> {row['맛집수']}개<br>
        <b>불금 유입 인구:</b> {int(row['총하차인원']):,}명<br>
        <b>혼잡 가성비 지수:</b> {row['혼잡가성비']:.2f}
    </div>
    """

    # Add circle markers to map (size proportional to shops, color reflects C/P ratio)
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=row['맛집수'] * 0.08, # 보기 좋은 크기로 스케일 조정
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(seoul_map)

# 코랩 화면에 지도 띄우기
seoul_map